# Part 2 — Notebook 01: Jet Clustering, Radius Dependence, & Subjet Exploration

Welcome to **Part 2** of the Experimental High-Energy Physics (HEP) Monte Carlo & Analysis Training Series!

In this notebook, you will transition from matrix-element generator truth partons to **reconstructed jets** — the observable, collimated sprays of hadrons produced when colored quarks and gluons shower and hadronize in a particle physics collision.

Using a 10-event Large Hadron Collider (LHC) $Z \to b\bar{b}$ dataset containing both detector-level (EMPFlow) and truth-level constituent 4-vectors, you will explore how sequential recombination jet algorithms ($anti-k_t$, $k_t$, Cambridge/Aachen) construct jets, analyze the impact of the radius parameter $R$, and recluster large-$R$ jets into subjets to reveal internal two-prong decay structure.


## Step 1: Environment Setup & Workspace Verification

Choose your execution environment using the `USE_COLAB` flag in the cell below:
- **Google Colab Mode (`USE_COLAB = True`)**: Mounts Google Drive at `/content/drive` for persistent storage and installs required HEP analysis packages (`uproot`, `awkward`, `vector`, `fastjet`, `mplhep`).
- **Local Machine Mode (`USE_COLAB = False`)**: Reads event data directly from `LOCAL_OUTPUT_DIR` (or defaults to `Part2_Jets/data/Zbb_RawConst.root` in your working directory) — no Google Drive or Colab dependencies required.

> [!IMPORTANT]
> **Dataset Prerequisite**: This notebook requires the dataset file `data/Zbb_RawConst.root`. Ensure the file is present in your working directory or persistent output workspace.


In [ ]:
import os
import sys
import shutil
import subprocess

# ── Execution Environment Mode Choice ──────────────────────────────────────────
# Set USE_COLAB = True when running on Google Colab to mount Google Drive.
# Set USE_COLAB = False when running locally on your own computer/laptop.
USE_COLAB = True  # <── Change to False if running locally on your machine

# For Local Mode (USE_COLAB = False), set LOCAL_OUTPUT_DIR to a base directory path,
# or leave as None to use the current working directory.
LOCAL_OUTPUT_DIR = None  # <── e.g., "/path/to/your/base/dir" or None

print("=== Part 2 Environment Verification & Setup ===")
print(f"Execution Mode : {'Google Colab' if USE_COLAB else 'Local Machine'}")
print(f"Python version : {sys.version.split()[0]}")

# ── 1. Configure Workspace Directory ─────────────────────────────────────────
if USE_COLAB:
    if os.path.exists("/content"):
        try:
            from google.colab import drive
            print("\nMounting Google Drive for persistent storage...")
            drive.mount('/content/drive', force_remount=False)
            output_dir = "/content/drive/MyDrive/MadGraph_Zbb_Outputs"
        except Exception as e:
            print(f"Notice: Drive mount skipped ({e}). Using local workspace directory.")
            output_dir = os.getcwd()
    else:
        output_dir = os.getcwd()
else:
    base_dir = os.path.abspath(os.path.expanduser(LOCAL_OUTPUT_DIR)) if LOCAL_OUTPUT_DIR else os.getcwd()
    output_dir = base_dir

os.makedirs(output_dir, exist_ok=True)
print(f"Persistent Working Directory: {os.getcwd()}")

# ── 2. Install Required HEP Packages ──────────────────────────────────────────
print("\nInstalling required HEP packages (uproot, awkward, vector, fastjet, mplhep)...")
!{sys.executable} -m pip install -q uproot awkward vector fastjet mplhep matplotlib numpy scipy

import numpy as np
import matplotlib.pyplot as plt
import mplhep as hep
import uproot
import awkward as ak
import vector
import fastjet

hep.style.use(hep.style.CMS)

# ── 3. Verify Dataset ROOT File ───────────────────────────────────────────────
dataset_file = os.path.join(os.getcwd(), "data/Zbb_RawConst.root")
if not os.path.exists(dataset_file):
    alt_path = os.path.join(os.getcwd(), "Zbb_RawConst.root")
    if os.path.exists(alt_path):
        dataset_file = alt_path
    else:
        print(f"Notice: ROOT file not found at default location '{dataset_file}'.")
        print("Please ensure 'Zbb_RawConst.root' is placed in the 'data/' directory.")

print(f"SUCCESS: Environment verified! Dataset path target: {dataset_file}")


## Step 2: Integrated Physics Foundations — Why Jets & Sequential Recombination

Before analyzing collider data or clustering particles, let's understand the physics of hadronic jets and sequential recombination algorithms.

### 2.1 Why Do We Need Jets?
In Quantum Chromodynamics (QCD), colored fundamental particles — **quarks** and **gluons** — carry color charge and cannot exist as isolated free particles (a phenomenon known as **color confinement**).

When a high-energy scattering process creates energetic quarks or gluons:
1. **Parton Showering**: The initial parton radiates gluons ($q \to qg$), which in turn split into quark-antiquark pairs ($g \to q\bar{q}$), producing a shower of partons.
2. **Hadronization**: As the system expands and cools below $\Lambda_{\text{QCD}} \approx 200\text{ MeV}$, color forces bind these partons into color-singlet hadrons (pions $\pi^\pm, \pi^0$, kaons $K$, protons $p$, neutrons $n$).
3. **Collimated Spray**: Because the initial parton possessed high transverse momentum $p_T$, the resulting daughter hadrons emerge tightly collimated along the direction of the original parton.

A **jet** is an algorithmically defined proxy for the underlying high-$p_T$ parton, constructed by grouping nearby final-state particles according to a mathematically reproducible rule.

---

### 2.2 Sequential Recombination Jet Algorithms
High-energy physics relies on **sequential recombination algorithms** to construct jets from a variable-length list of constituent 4-vectors. These algorithms compute two distance metrics for all particles/clusters in an event:

1. **Pair Distance** ($d_{ij}$): Distance between pseudo-jets $i$ and $j$:
   $$d_{ij} = \min\left(p_{T,i}^{2p}, p_{T,j}^{2p}\right) \frac{\Delta R_{ij}^2}{R^2}$$
2. **Beam Distance** ($d_{iB}$): Distance between pseudo-jet $i$ and the collider beam axis:
   $$d_{iB} = p_{T,i}^{2p}$$

Where $\Delta R_{ij}^2 = (\eta_i - \eta_j)^2 + (\phi_i - \phi_j)^2$ is the angular separation, $R$ is the jet radius parameter, and $p$ determines the algorithmic power:

| Algorithm Name | Power Parameter ($p$) | Distance Metric $d_{ij}$ Feature | Clustered Jet Shape Characteristics |
| :--- | :---: | :--- | :--- |
| **anti-$k_t$** | $p = -1$ | Hard particles ($p_{T,i}^{-2}$) recombine first | Forms stable, rigid, circular boundaries around hard cores |
| **$k_t$** | $p = +1$ | Soft particles ($p_{T,i}^{+2}$) recombine first | Irregular boundaries; useful for QCD subjets and clustering history |
| **Cambridge/Aachen (C/A)** | $p = 0$ | Pure angular distance $\Delta R_{ij}^2 / R^2$ | Purely geometrical proximity; ideal for boosted decay reclustering |

#### The Iterative Algorithm Loop:
1. Compute all $d_{ij}$ and $d_{iB}$ for the current list of objects.
2. Find the global minimum distance $d_{\text{min}} = \min(d_{ij}, d_{iB})$.
3. If $d_{\text{min}}$ is a pair distance $d_{ij}$, combine objects $i$ and $j$ into a single 4-vector $p_{ij}^\mu = p_i^\mu + p_j^\mu$, remove $i$ and $j$, and return to Step 1.
4. If $d_{\text{min}}$ is a beam distance $d_{iB}$, declare object $i$ a **completed jet**, remove it from the active list, and return to Step 1.
5. Stop when no active objects remain.

---

### 2.3 Radius Parameter $R$
The parameter $R$ defines the effective angular clustering radius in $(\eta, \phi)$ space:
- **Small-$R$ Jets ($R = 0.4$)**: Standard scale for resolving individual quarks, gluons, and isolated decay products while minimizing contamination from background radiation.
- **Large-$R$ Jets ($R = 0.8$ or $R = 1.0$)**: Wide scale designed to capture all collimated decay products of a high-$p_T$ boosted heavy resonance (like $Z \to b\bar{b}$ or $W \to q\bar{q}$) inside a single merged jet.

---

### 2.4 EMPFlow vs. Truth Constituents
The ROOT dataset `Zbb_RawConst.root` contains two sets of event constituents:
- **`constituents_EMPFlow_*`**: Detector-level Energy-Matched Particle Flow objects (charged tracks matched to inner detector measurements + neutral calorimeter topological clusters).
- **`constituents_Truth_*`**: Generator-level truth particles prior to detector interaction.

For particle identification, refer to the official [PDG Monte Carlo Particle Numbering Scheme](https://pdg.lbl.gov/2007/reviews/montecarlorpp.pdf).

---

> [!IMPORTANT]
> ### Self-Reflection Checkpoint 2.1
> **Conceptual Question**: Why is a jet algorithm needed instead of simply calling every final-state particle a jet?

<details>
<summary>Click to show Checkpoint 2.1 Reference Solution</summary>

<p>Final-state particles (pions, photons, protons) are individual quantum states produced during hadronization. A single high-$p_T$ quark or gluon produces tens to hundreds of these particles. Calling each particle a jet would miss the underlying hard-scattering kinematics ($p_T, m, \eta$) of the parent parton. Jet algorithms combine these variable-length sprays into stable, reproducible 4-vectors that directly correspond to the hard-scattered partons.</p>

</details>


## Step 3: Loading ROOT Event Data & Single-Event Constituent Display

Now, let's load the 10-event $Z \to b\bar{b}$ dataset using `uproot` and inspect the constituent 4-vectors ($p_T, \eta, \phi, m, e, \text{pdgId}, \text{charge}$).

We will visualize a single event in the $(\eta, \phi)$ plane, representing constituent transverse momentum $p_T$ via marker size and color while handling periodic azimuthal angle wrapping $\phi \in [-\pi, \pi]$.


In [ ]:
# ── Function to load ROOT event tree ──────────────────────────────────────────
def load_zbb_dataset(filepath):
    if not os.path.exists(filepath):
        print(f"File {filepath} does not exist. Creating synthetic demonstration event data...")
        events = []
        for ev in range(10):
            n_const = np.random.randint(40, 90)
            pt = np.random.exponential(scale=15.0, size=n_const) + 1.0
            eta = np.random.uniform(-2.5, 2.5, size=n_const)
            phi = np.random.uniform(-np.pi, np.pi, size=n_const)
            m = np.full(n_const, 0.14)
            e = np.sqrt(pt**2 * np.cosh(eta)**2 + m**2)
            pdgId = np.random.choice([211, -211, 22, 2112, 2212], size=n_const)
            charge = np.where(np.abs(pdgId)==211, 1.0, 0.0)
            events.append({
                "pt": pt, "eta": eta, "phi": phi, "m": m, "e": e, "pdgId": pdgId, "charge": charge
            })
        return events

    with uproot.open(filepath) as f:
        tree = f["analysis"]
        arrays = tree.arrays([
            "constituents_EMPFlow_pt", "constituents_EMPFlow_eta",
            "constituents_EMPFlow_phi", "constituents_EMPFlow_m",
            "constituents_EMPFlow_e", "constituents_EMPFlow_pdgId",
            "constituents_EMPFlow_charge"
        ], library="ak")
        
        events = []
        for i in range(len(arrays)):
            pt_raw = ak.to_numpy(arrays["constituents_EMPFlow_pt"][i])
            m_raw = ak.to_numpy(arrays["constituents_EMPFlow_m"][i])
            e_raw = ak.to_numpy(arrays["constituents_EMPFlow_e"][i])
            
            # Check unit scale (MeV vs GeV)
            scale = 1000.0 if np.mean(pt_raw) > 500 else 1.0
            
            events.append({
                "pt": pt_raw / scale,
                "eta": ak.to_numpy(arrays["constituents_EMPFlow_eta"][i]),
                "phi": ak.to_numpy(arrays["constituents_EMPFlow_phi"][i]),
                "m": m_raw / scale,
                "e": e_raw / scale,
                "pdgId": ak.to_numpy(arrays["constituents_EMPFlow_pdgId"][i]),
                "charge": ak.to_numpy(arrays["constituents_EMPFlow_charge"][i])
            })
        return events

# ── Function for periodic phi difference ─────────────────────────────────────
def delta_phi(phi1, phi2):
    dphi = phi1 - phi2
    return np.arctan2(np.sin(dphi), np.cos(dphi))

# Load events
events_empflow = load_zbb_dataset(dataset_file)
print(f"Loaded {len(events_empflow)} events from dataset.")

# Plot Event 0 constituents in (eta, phi) plane
ev0 = events_empflow[0]
fig, ax = plt.subplots(figsize=(10, 6))
sc = ax.scatter(
    ev0["eta"], ev0["phi"],
    s=ev0["pt"] * 10.0,
    c=ev0["pt"],
    cmap="plasma",
    alpha=0.85,
    edgecolors="k",
    linewidths=0.5
)
cbar = plt.colorbar(sc, ax=ax)
cbar.set_label("Constituent $p_T$ [GeV]")

ax.set_xlabel("Pseudorapidity $\eta$")
ax.set_ylabel("Azimuthal Angle $\phi$ [rad]")
ax.set_title("Event 0: Detector-Level EMPFlow Constituents in $(\eta, \phi)$ Plane")
ax.set_ylim(-np.pi, np.pi)
ax.grid(True, linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()

print(f"Event 0 Constituent Count: {len(ev0['pt'])}")
print(f"Max Constituent pT       : {np.max(ev0['pt']):.2f} GeV")


---

> [!IMPORTANT]
> ### Exercise 3a: Constituent Multiplicity & Transverse Momentum Inspection
> Write code in the cell below to inspect **Event 0** and **Event 1**:
> 1. Compute and print the total constituent count $N_{\text{const}}$ for each event.
> 2. Compute and print the scalar sum of constituent transverse momentum $\sum p_T$ for each event.


In [ ]:
# ── TODO: Write code to inspect Event 0 and Event 1 constituent metrics ──
ev0 = events_empflow[0]
ev1 = events_empflow[1]

# TODO: Compute N_const and sum(pT) for Event 0
n_const_ev0 = len(ev0["pt"])
sum_pt_ev0 = np.sum(ev0["pt"])

# TODO: Compute N_const and sum(pT) for Event 1
n_const_ev1 = len(ev1["pt"])
sum_pt_ev1 = np.sum(ev1["pt"])

print(f"Event 0: N_const = {n_const_ev0}, Scalar sum pT = {sum_pt_ev0:.2f} GeV")
print(f"Event 1: N_const = {n_const_ev1}, Scalar sum pT = {sum_pt_ev1:.2f} GeV")


<details>
<summary>Click to show Exercise 3a Reference Solution</summary>

```python
ev0 = events_empflow[0]
ev1 = events_empflow[1]

print(f"Event 0: N_const = {len(ev0['pt'])}, sum(pT) = {np.sum(ev0['pt']):.2f} GeV")
print(f"Event 1: N_const = {len(ev1['pt'])}, sum(pT) = {np.sum(ev1['pt']):.2f} GeV")
```

</details>

---


## Step 4: Jet Clustering with FastJet & Algorithm Comparison

Now, let's convert constituent 4-vectors into `fastjet.PseudoJet` objects and run anti-$k_t$ clustering at $R = 0.4$.

We will overlay the clustered jet centers and angular boundaries over the $(\eta, \phi)$ constituent scatter plot, and compare anti-$k_t$, $k_t$, and Cambridge/Aachen (C/A) jet shapes side-by-side.


In [ ]:
# ── Helper function to cluster an event with FastJet ─────────────────────────
def cluster_event(event_dict, algo_name="antikt", R=0.4, pt_min=20.0):
    pj_list = []
    for i in range(len(event_dict["pt"])):
        pt = event_dict["pt"][i]
        eta = event_dict["eta"][i]
        phi = event_dict["phi"][i]
        m = event_dict["m"][i]
        px = pt * np.cos(phi)
        py = pt * np.sin(phi)
        pz = pt * np.sinh(eta)
        e = np.sqrt(px**2 + py**2 + pz**2 + m**2)
        pj = fastjet.PseudoJet(px, py, pz, e)
        pj.set_user_index(i)
        pj_list.append(pj)

    if algo_name.lower() in ["antikt", "anti-kt", "anti_kt"]:
        jet_def = fastjet.JetDefinition(fastjet.antikt_algorithm, R)
    elif algo_name.lower() in ["kt"]:
        jet_def = fastjet.JetDefinition(fastjet.kt_algorithm, R)
    elif algo_name.lower() in ["ca", "cambridge", "cambridge_aachen"]:
        jet_def = fastjet.JetDefinition(fastjet.cambridge_algorithm, R)
    else:
        raise ValueError(f"Unknown algorithm: {algo_name}")

    cluster_seq = fastjet.ClusterSequence(pj_list, jet_def)
    inclusive_jets = cluster_seq.inclusive_jets(pt_min)
    sorted_jets = sorted(inclusive_jets, key=lambda j: j.pt(), reverse=True)
    return sorted_jets, cluster_seq

# Cluster Event 0 with anti-kt R=0.4
jets_antikt_04, _ = cluster_event(events_empflow[0], algo_name="antikt", R=0.4, pt_min=20.0)

print(f"=== Event 0 Clustered Jets (anti-kt, R=0.4, pT > 20 GeV) ===")
print(f"Number of reconstructed jets: {len(jets_antikt_04)}")
for idx, j in enumerate(jets_antikt_04):
    print(f"  Jet {idx}: pT = {j.pt():6.2f} GeV, eta = {j.eta():5.2f}, phi = {j.phi():5.2f}, mass = {j.m():6.2f} GeV, n_const = {len(j.constituents())}")

# ── Plot side-by-side comparison of anti-kt, kt, and C/A ─────────────────────
algos = [("anti-kt", "antikt"), ("$k_t$", "kt"), ("Cambridge/Aachen", "ca")]
fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=True)

ev0 = events_empflow[0]

for ax_idx, (label, algo_key) in enumerate(algos):
    ax = axes[ax_idx]
    jets, _ = cluster_event(ev0, algo_name=algo_key, R=0.4, pt_min=20.0)
    
    ax.scatter(ev0["eta"], ev0["phi"], s=ev0["pt"]*6.0, c="gray", alpha=0.5, label="Constituents")
    
    for j_idx, j in enumerate(jets):
        circle = plt.Circle((j.eta(), j.phi()), 0.4, color=f"C{j_idx}", fill=False, linewidth=2, linestyle="--")
        ax.add_patch(circle)
        ax.plot(j.eta(), j.phi(), marker="x", color=f"C{j_idx}", markersize=10, markeredgewidth=3, label=f"Jet {j_idx} ($p_T={j.pt():.1f}$ GeV)")
    
    ax.set_title(f"Algorithm: {label} ($R=0.4$)")
    ax.set_xlabel("Pseudorapidity $\eta$")
    if ax_idx == 0:
        ax.set_ylabel("Azimuthal Angle $\phi$ [rad]")
    ax.set_ylim(-np.pi, np.pi)
    ax.grid(True, linestyle="--", alpha=0.4)
    ax.legend(loc="upper right", fontsize=8)

plt.tight_layout()
plt.show()


---

> [!IMPORTANT]
> ### Exercise 4b: Constituent Merging Prediction
> **Conceptual Question**: Consider two hard particles separated by angular distance $\Delta R = 0.3$ in $(\eta, \phi)$. Will they be merged into a single jet when clustered with $anti-k_t$ at $R = 0.4$? What about at $R = 0.2$?

<details>
<summary>Click to show Exercise 4b Reference Solution</summary>

<p>For $anti-k_t$ clustering, particles separated by $\Delta R < R$ fall within each other's active attraction cone.</p>
<ul>
<li><b>At $R = 0.4$</b>: Since $\Delta R = 0.3 < 0.4$, the hard pair distance $d_{12}$ is smaller than the beam distance $d_{1B}$, so they <b>will be merged</b> into a single jet.</li>
<li><b>At $R = 0.2$</b>: Since $\Delta R = 0.3 > 0.2$, the beam distance $d_{iB}$ is smaller than the pair distance $d_{12}$, so they <b>will not be merged</b> and will form two separate small-$R$ jets.</li>
</ul>

</details>

---


## Step 5: Radius Scan ($R=0.2, 0.4, 0.8, 1.0$) with $Z \to b\bar{b}$ Events

Now, let's systematically scan the jet radius parameter across $R \in \{0.2, 0.4, 0.8, 1.0\}$ for all 10 dataset events, keeping a fixed minimum jet threshold of $p_T > 20\text{ GeV}$.

We will produce:
1. An event-by-event table of jet multiplicities for each $R$.
2. Side-by-side $(\eta, \phi)$ displays comparing small-$R$ ($R=0.4$) vs. large-$R$ ($R=1.0$) jet containment.
3. Distributions of leading jet $p_T$, $\eta$, and mass across the dataset.


In [ ]:
# ── Radius Scan Loop ──────────────────────────────────────────────────────────
radii = [0.2, 0.4, 0.8, 1.0]
results_by_R = {R: [] for R in radii}

print(f"{'Event':<7s} | {'R=0.2 Jets':<10s} | {'R=0.4 Jets':<10s} | {'R=0.8 Jets':<10s} | {'R=1.0 Jets':<10s}")
print("-" * 58)

for ev_idx, ev in enumerate(events_empflow):
    row_str = f"{ev_idx:<7d} | "
    for R in radii:
        jets, _ = cluster_event(ev, algo_name="antikt", R=R, pt_min=20.0)
        results_by_R[R].append(jets)
        row_str += f"{len(jets):<10d} | "
    print(row_str)

# ── Side-by-Side Event 0 Display: R=0.4 vs R=1.0 ──────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6), sharey=True)

ev0 = events_empflow[0]
jets_04 = results_by_R[0.4][0]
jets_10 = results_by_R[1.0][0]

ax1.scatter(ev0["eta"], ev0["phi"], s=ev0["pt"]*6.0, c="gray", alpha=0.5)
for idx, j in enumerate(jets_04):
    circle = plt.Circle((j.eta(), j.phi()), 0.4, color=f"C{idx}", fill=False, linewidth=2, linestyle="--")
    ax1.add_patch(circle)
    ax1.plot(j.eta(), j.phi(), marker="x", color=f"C{idx}", markersize=10, markeredgewidth=2, label=f"Jet {idx} ($p_T={j.pt():.1f}$ GeV)")
ax1.set_title("Small-$R$ Jet Clustering ($R=0.4$)")
ax1.set_xlabel("Pseudorapidity $\eta$")
ax1.set_ylabel("Azimuthal Angle $\phi$ [rad]")
ax1.set_ylim(-np.pi, np.pi)
ax1.grid(True, linestyle="--", alpha=0.4)
ax1.legend(loc="upper right", fontsize=8)

ax2.scatter(ev0["eta"], ev0["phi"], s=ev0["pt"]*6.0, c="gray", alpha=0.5)
for idx, j in enumerate(jets_10):
    circle = plt.Circle((j.eta(), j.phi()), 1.0, color=f"C{idx}", fill=False, linewidth=2, linestyle="-")
    ax2.add_patch(circle)
    ax2.plot(j.eta(), j.phi(), marker="X", color=f"C{idx}", markersize=12, markeredgewidth=2, label=f"Large-$R$ Jet {idx} ($m={j.m():.1f}$ GeV)")
ax2.set_title("Large-$R$ Jet Clustering ($R=1.0$)")
ax2.set_xlabel("Pseudorapidity $\eta$")
ax2.set_ylim(-np.pi, np.pi)
ax2.grid(True, linestyle="--", alpha=0.4)
ax2.legend(loc="upper right", fontsize=8)

plt.tight_layout()
plt.show()

# ── Distribution Plots: Leading Jet Mass vs R ─────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

for R in radii:
    lead_pts = [jets[0].pt() for jets in results_by_R[R] if len(jets) > 0]
    lead_masses = [jets[0].m() for jets in results_by_R[R] if len(jets) > 0]
    
    ax1.hist(lead_pts, bins=np.linspace(0, 300, 15), histtype="step", linewidth=2, label=f"$R={R}$")
    ax2.hist(lead_masses, bins=np.linspace(0, 150, 15), histtype="step", linewidth=2, label=f"$R={R}$")

ax1.set_xlabel("Leading Jet $p_T$ [GeV]")
ax1.set_ylabel("Events")
ax1.set_title("Leading Jet Transverse Momentum Spectrum")
ax1.grid(True, linestyle="--", alpha=0.4)
ax1.legend()

ax2.set_xlabel("Leading Jet Mass $m_{\text{jet}}$ [GeV]")
ax2.set_ylabel("Events")
ax2.set_title("Leading Jet Mass Spectrum across Radius $R$")
ax2.grid(True, linestyle="--", alpha=0.4)
ax2.legend()

plt.tight_layout()
plt.show()


---

> [!IMPORTANT]
> ### Self-Reflection & Physics Checkpoint 5.1
> **Conceptual Question**: What is one benefit and one cost of increasing the jet radius parameter $R$ from 0.4 to 1.0?

<details>
<summary>Click to show Checkpoint 5.1 Reference Solution</summary>

<ul>
<li><b>Benefit of $R=1.0$</b>: Captures all wide-angle decay products and gluon radiation from boosted heavy resonance decays (like $Z \to b\bar{b}$), reconstructing the full resonance mass inside a single large-$R$ jet.</li>
<li><b>Cost of $R=1.0$</b>: Collects significantly more soft background radiation, underlying event (UE) activity, and pileup interactions, which broadens jet energy resolution and increases susceptibility to background contamination.</li>
</ul>

</details>

---


## Step 6: Large-$R$ ($R=1.0$) Subjet Reclustering & Boosted $Z \to b\bar{b}$ Structure

Now, let's explore **jet substructure**. When a high-$p_T$ $Z$ boson decays into a bottom-antibottom pair ($Z \to b\bar{b}$), both $b$-quarks fall inside a single $R=1.0$ large-$R$ jet.

To expose the internal two-prong decay structure:
1. Reconstruct large-$R$ jets with anti-$k_t$ ($R=1.0$).
2. Select the leading large-$R$ jet ($p_T > 150\text{ GeV}$).
3. Extract its constituents.
4. **Recluster** those constituents using exclusive $k_t$ algorithm with $N = 2$ subjets.
5. Compute subjet observables: subjet $p_T$, subjet mass, and subjet angular separation $\Delta R_{\text{subjet}}$.


In [ ]:
# ── Recluster Leading Large-R Jet into N=2 Subjets ────────────────────────────
subjet_results = []

for ev_idx, ev in enumerate(events_empflow):
    large_jets, _ = cluster_event(ev, algo_name="antikt", R=1.0, pt_min=100.0)
    if len(large_jets) == 0:
        continue
    
    lead_large_jet = large_jets[0]
    consts = lead_large_jet.constituents()
    
    subjet_def = fastjet.JetDefinition(fastjet.kt_algorithm, 1.0)
    sub_seq = fastjet.ClusterSequence(consts, subjet_def)
    exclusive_subjets = sub_seq.exclusive_jets(2)
    
    sorted_subjets = sorted(exclusive_subjets, key=lambda sj: sj.pt(), reverse=True)
    
    if len(sorted_subjets) == 2:
        sj1, sj2 = sorted_subjets[0], sorted_subjets[1]
        dr_sub = np.sqrt((sj1.eta() - sj2.eta())**2 + delta_phi(sj1.phi(), sj2.phi())**2)
        subjet_results.append({
            "event": ev_idx,
            "large_pt": lead_large_jet.pt(),
            "large_m": lead_large_jet.m(),
            "sj1_pt": sj1.pt(),
            "sj2_pt": sj2.pt(),
            "dr_sub": dr_sub
        })

print(f"Reclustered N=2 subjets for {len(subjet_results)} high-pT events.")
print(f"{'Event':<6s} | {'Large Jet pT':<13s} | {'Large Jet Mass':<14s} | {'Subjet 1 pT':<12s} | {'Subjet 2 pT':<12s} | {'Subjet dR':<10s}")
print("-" * 78)
for res in subjet_results:
    print(f"{res['event']:<6d} | {res['large_pt']:11.2f} GeV | {res['large_m']:12.2f} GeV | {res['sj1_pt']:10.2f} GeV | {res['sj2_pt']:10.2f} GeV | {res['dr_sub']:8.3f}")

if len(subjet_results) > 0:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    drs = [res["dr_sub"] for res in subjet_results]
    large_pts = [res["large_pt"] for res in subjet_results]
    
    ax1.scatter(large_pts, drs, color="crimson", s=60, edgecolors="k")
    pt_grid = np.linspace(100, 350, 100)
    dr_guide = 2.0 * 91.2 / pt_grid
    ax1.plot(pt_grid, dr_guide, "k--", label="Theoretical Boost Guide $\Delta R \approx 2 m_Z / p_T^Z$")
    
    ax1.set_xlabel("Large-$R$ Jet $p_T$ [GeV]")
    ax1.set_ylabel("Subjet Angular Separation $\Delta R_{\text{subjet}}$")
    ax1.set_title("Subjet Angular Separation vs. Large-$R$ Transverse Momentum")
    ax1.grid(True, linestyle="--", alpha=0.4)
    ax1.legend()

    sj1_pts = [res["sj1_pt"] for res in subjet_results]
    sj2_pts = [res["sj2_pt"] for res in subjet_results]
    ax2.hist(sj1_pts, bins=10, alpha=0.7, label="Leading Subjet $p_{T,1}$")
    ax2.hist(sj2_pts, bins=10, alpha=0.7, label="Subleading Subjet $p_{T,2}$")
    ax2.set_xlabel("Subjet Transverse Momentum [GeV]")
    ax2.set_ylabel("Events")
    ax2.set_title("Reclustered Subjet $p_T$ Spectra ($N=2$)")
    ax2.grid(True, linestyle="--", alpha=0.4)
    ax2.legend()

    plt.tight_layout()
    plt.show()


---

> [!IMPORTANT]
> ### Self-Reflection & Physics Checkpoint 6.1
> **Conceptual Question**: Why does observing two subjets inside a large-$R$ jet not establish by itself that the jet originated from a $Z \to b\bar{b}$ decay?

<details>
<summary>Click to show Checkpoint 6.1 Reference Solution</summary>

<p>Reclustering a large-$R$ jet into $N=2$ subjets simply enforces a two-body momentum division. Pure QCD background jets produced by gluon splitting ($g \to q\bar{q}$) or hard asymmetric gluon radiation also exhibit two-prong substructure. Confirming a true $Z \to b\bar{b}$ decay requires invariant mass consistency ($m_{\text{jet}} \approx m_Z = 91.2\text{ GeV}$) combined with $b$-tagging algorithms to verify bottom-quark flavor origin.</p>

</details>

---


## Step 7: Truth-Level vs. EMPFlow Detector-Level Comparison

Finally, let's compare **truth-level particles** (`constituents_Truth_*`) against **detector-level EMPFlow objects** (`constituents_EMPFlow_*`).

This comparison illustrates how hadronization, neutral particle loss, and calorimeter cluster merging alter constituent multiplicities, jet energy scales, and reconstructed jet masses.


In [ ]:
# ── Load Truth-level constituents ─────────────────────────────────────────────
def load_truth_dataset(filepath):
    if not os.path.exists(filepath):
        return load_zbb_dataset(filepath)

    with uproot.open(filepath) as f:
        tree = f["analysis"]
        arrays = tree.arrays([
            "constituents_Truth_pt", "constituents_Truth_eta",
            "constituents_Truth_phi", "constituents_Truth_m",
            "constituents_Truth_e", "constituents_Truth_pdgId",
            "constituents_Truth_charge"
        ], library="ak")
        
        events = []
        for i in range(len(arrays)):
            pt_raw = ak.to_numpy(arrays["constituents_Truth_pt"][i])
            m_raw = ak.to_numpy(arrays["constituents_Truth_m"][i])
            e_raw = ak.to_numpy(arrays["constituents_Truth_e"][i])
            scale = 1000.0 if np.mean(pt_raw) > 500 else 1.0
            
            events.append({
                "pt": pt_raw / scale,
                "eta": ak.to_numpy(arrays["constituents_Truth_eta"][i]),
                "phi": ak.to_numpy(arrays["constituents_Truth_phi"][i]),
                "m": m_raw / scale,
                "e": e_raw / scale,
                "pdgId": ak.to_numpy(arrays["constituents_Truth_pdgId"][i]),
                "charge": ak.to_numpy(arrays["constituents_Truth_charge"][i])
            })
        return events

events_truth = load_truth_dataset(dataset_file)

truth_nconst = [len(ev["pt"]) for ev in events_truth]
emp_nconst = [len(ev["pt"]) for ev in events_empflow]

truth_jet_masses = []
emp_jet_masses = []

for i in range(len(events_truth)):
    j_t, _ = cluster_event(events_truth[i], algo_name="antikt", R=1.0, pt_min=50.0)
    j_e, _ = cluster_event(events_empflow[i], algo_name="antikt", R=1.0, pt_min=50.0)
    if len(j_t) > 0 and len(j_e) > 0:
        truth_jet_masses.append(j_t[0].m())
        emp_jet_masses.append(j_e[0].m())

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.hist(truth_nconst, bins=10, alpha=0.7, label="Truth Particles", color="forestgreen")
ax1.hist(emp_nconst, bins=10, alpha=0.7, label="EMPFlow Detector Objects", color="navy")
ax1.set_xlabel("Event Constituent Multiplicity $N_{\text{const}}$")
ax1.set_ylabel("Events")
ax1.set_title("Constituent Multiplicity: Truth vs. EMPFlow")
ax1.grid(True, linestyle="--", alpha=0.4)
ax1.legend()

if len(truth_jet_masses) > 0:
    ax2.hist(truth_jet_masses, bins=8, alpha=0.7, label="Truth Large-$R$ Jet Mass", color="forestgreen")
    ax2.hist(emp_jet_masses, bins=8, alpha=0.7, label="EMPFlow Large-$R$ Jet Mass", color="navy")
    ax2.set_xlabel("Leading Large-$R$ Jet Mass $m_{\text{jet}}$ [GeV]")
    ax2.set_ylabel("Events")
    ax2.set_title("Reconstructed Jet Mass: Truth vs. EMPFlow ($R=1.0$)")
    ax2.grid(True, linestyle="--", alpha=0.4)
    ax2.legend()

plt.tight_layout()
plt.show()

print("=== Truth vs EMPFlow Comparison Summary ===")
print(f"Mean Truth Constituent Count : {np.mean(truth_nconst):.1f}")
print(f"Mean EMPFlow Constituent Count: {np.mean(emp_nconst):.1f}")


---

## Summary & Next Steps

Congratulations! You have completed **Part 2: Jet Clustering, Radius Dependence, & Subjet Exploration**.

### Key Concepts Mastered:
1. **Jets as Algorithmic Objects**: Quarks and gluons shower and hadronize; jet algorithms ($anti-k_t$, $k_t$, Cambridge/Aachen) group constituent sprays into reproducible 4-vectors.
2. **Sequential Recombination Mechanics**: Distance metrics $d_{ij}$ and $d_{iB}$ determine recombination order; $anti-k_t$ forms stable, rigid, circular boundaries around hard cores.
3. **Radius Parameter $R$**: Small-$R$ ($R=0.4$) resolves individual partons, while large-$R$ ($R=1.0$) captures collimated boosted decays ($Z \to b\bar{b}$) at the cost of increased pileup/UE inclusion.
4. **Subjet Substructure**: Reclustering $R=1.0$ large-$R$ jets into $N=2$ subjets exposes two-prong momentum division consistent with boosted hadronic resonance decays.
5. **Truth vs. Detector Constituents**: Hadronization and detector resolution shift constituent multiplicities and jet mass spectra.

**Proceed to Part 3 (Detector-Level ROOT Analysis, Machine Learning Taggers, & MC Normalization)** to apply machine learning jet taggers and perform detector-level ROOT analysis!
